# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CloversXisel/Flyrank-Internship2026/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule

I will prioritize content items for human review when they have enough search visibility to make the review worthwhile. Among those items, I will give higher priority to pages that have not been updated recently, have a low click-through rate (CTR), or have an average search position between 4 and 20.

The score is a transparent, rule-based decision-support baseline. It does not predict whether a content refresh will improve performance. The thresholds are fixed before evaluation and are not fitted to the outcome label.

### Reason codes

- `visible_stale`: The item has sufficient impressions and has not been updated for at least 180 days.
- `visible_low_ctr`: The item has sufficient impressions and CTR is below 1.0%.
- `visible_position_opportunity`: The item has sufficient impressions and average position is between 4 and 20.
- `visible_no_clear_issue`: The item has sufficient impressions, but none of the three review signals is present.
- `low_visibility`: The item has fewer than 500 impressions.
- `missing_metrics`: One or more required metrics are unavailable.

The dataset stores rate columns as percentage points multiplied by 100, so a CTR value of `0.76` means 0.76%, not 76%. An average position of `0` means missing data, not rank zero.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Locate the anonymized starter dataset
candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None:
    try:
        from google.colab import files

        print("Upload content_refresh_anonymized.csv")
        uploaded = files.upload()

        csv_files = [
            name for name in uploaded
            if name.lower().endswith(".csv")
        ]

        if not csv_files:
            raise FileNotFoundError("No CSV file was uploaded.")

        data_path = Path(csv_files[0])

    except ImportError:
        raise FileNotFoundError(
            "Dataset not found. Place it at "
            "data/raw/content_refresh_anonymized.csv."
        )

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())

Upload content_refresh_anonymized.csv


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Scoring methodology

The baseline uses three simple signals: content freshness, click-through rate, and average search position. A page must have at least 500 observed impressions to receive a positive action score.

The score is the natural logarithm of one plus impressions, multiplied by the number of review signals present. This rewards pages with multiple issues while reducing the dominance of extremely high impression counts.

The score uses fixed thresholds and no fitted weights. Pseudonymous IDs are used only for identification and tie-breaking, never as predictive features. Trend fields, product flags, and future-window labels are excluded from the score.

In [ ]:
# Identify the columns available in the dataset.
def find_column(candidates, columns):
    for col in candidates:
        if col in columns:
            return col
    return None


impressions_col = find_column(
    ["impressions_90d", "impressions"], df.columns
)

ctr_col = find_column(
    ["ctr"], df.columns
)

position_col = find_column(
    ["avg_position", "gsc_avg_position"], df.columns
)

update_age_col = find_column(
    [
        "days_since_update",
        "days_since_last_update",
        "days_since_content_update",
    ],
    df.columns,
)

content_id_col = find_column(
    ["content_id"], df.columns
)

required = {
    "impressions": impressions_col,
    "CTR": ctr_col,
    "average position": position_col,
    "days since update": update_age_col,
}

missing = [
    name for name, col in required.items()
    if col is None
]

if missing:
    raise KeyError(
        f"Missing required columns: {missing}. "
        "Check docs/data-dictionary.md and update the "
        "column candidates to match the actual dataset."
    )


# Work on a separate copy.
queue = df.copy()

queue["_impressions"] = pd.to_numeric(
    queue[impressions_col], errors="coerce"
)

queue["_ctr"] = pd.to_numeric(
    queue[ctr_col], errors="coerce"
)

queue["_position"] = pd.to_numeric(
    queue[position_col], errors="coerce"
)

queue["_update_age"] = pd.to_numeric(
    queue[update_age_col], errors="coerce"
)


# Dataset-specific thresholds.
MIN_IMPRESSIONS = 500
STALE_DAYS = 180
LOW_CTR = 1.0
POSITION_MIN = 4
POSITION_MAX = 20


# avg_position = 0 means missing data.
queue.loc[
    queue["_position"] == 0, "_position"
] = np.nan


# Build the review signals.
queue["signal_stale"] = (
    queue["_update_age"] >= STALE_DAYS
).fillna(False)

queue["signal_low_ctr"] = (
    queue["_ctr"] < LOW_CTR
).fillna(False)

queue["signal_position_opportunity"] = (
    queue["_position"].between(
        POSITION_MIN,
        POSITION_MAX,
        inclusive="both",
    )
).fillna(False)

queue["signal_visible"] = (
    queue["_impressions"] >= MIN_IMPRESSIONS
).fillna(False)


# Count the number of review signals.
queue["signal_count"] = (
    queue["signal_stale"].astype(int)
    + queue["signal_low_ctr"].astype(int)
    + queue["signal_position_opportunity"].astype(int)
)


# Score: visible pages only.
queue["baseline_action_score"] = (
    np.log1p(
        queue["_impressions"].clip(lower=0).fillna(0)
    )
    * queue["signal_count"]
    * queue["signal_visible"].astype(int)
)


# Assign reason codes.
def assign_reason(row):
    if not row["signal_visible"]:
        return "low_visibility"

    if (
        pd.isna(row["_ctr"])
        or pd.isna(row["_position"])
        or pd.isna(row["_update_age"])
    ):
        if row["signal_count"] > 0:
            return "missing_metrics_with_review_signal"
        return "missing_metrics"

    reasons = []

    if row["signal_stale"]:
        reasons.append("visible_stale")

    if row["signal_low_ctr"]:
        reasons.append("visible_low_ctr")

    if row["signal_position_opportunity"]:
        reasons.append("visible_position_opportunity")

    if not reasons:
        return "visible_no_clear_issue"

    return "|".join(reasons)


queue["reason_code"] = queue.apply(
    assign_reason, axis=1
)


# Assign a suggested action.
queue["action"] = np.where(
    queue["baseline_action_score"] > 0,
    "Review for possible content refresh",
    "Monitor or deprioritize",
)


# Rank by score, then impressions.
sort_columns = [
    "baseline_action_score",
    "_impressions",
]

ascending_order = [False, False]

if content_id_col:
    sort_columns.append(content_id_col)
    ascending_order.append(True)

queue = queue.sort_values(
    sort_columns,
    ascending=ascending_order,
    na_position="last",
).reset_index(drop=True)

queue.insert(
    0,
    "baseline_rank",
    np.arange(1, len(queue) + 1),
)


# Export only safe review fields.
export_columns = []

if content_id_col:
    export_columns.append(content_id_col)

export_columns.extend([
    impressions_col,
    ctr_col,
    position_col,
    update_age_col,
    "baseline_action_score",
    "baseline_rank",
    "action",
    "reason_code",
])

export_columns = list(dict.fromkeys(export_columns))

output = queue[export_columns].copy()


# Check that private or identifying text fields are excluded.
forbidden_terms = [
    "client_name",
    "url",
    "query",
    "keyword",
    "title",
    "domain",
]

unsafe_columns = [
    col for col in output.columns
    if any(term in col.lower() for term in forbidden_terms)
]

if unsafe_columns:
    raise ValueError(
        f"Unsafe columns in export: {unsafe_columns}"
    )


# Write the ranked CSV.
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

output.to_csv(output_path, index=False)

print(f"Ranked {len(output):,} items.")
print(f"CSV saved to: {output_path}")

display(output.head(10))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.